# Data Exploration and Preprocessing

We have chosen to use the video games category for this project.

## Imports

In [1]:
import pandas as pd
import json

## Load the data

In [2]:
file_dict = {}
files = {'user_reviews' : '../data/raw/Video_Games.jsonl',
         'meta_data' : '../data/raw/meta_Video_Games.jsonl'}
for name,file in files.items():
    record_list = []
    with open(file, 'r') as fp:
        for line in fp:
            record = json.loads(line.strip())
            record_list.append(record)
            
            
    file_dict[name]=record_list

## User reviews

### Description

In [3]:
user_reviews = pd.DataFrame(file_dict['user_reviews'])
user_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4.0,It’s pretty sexual. Not my fav,I’m playing on ps5 and it’s interesting. It’s...,[],B07DJWBYKP,B07DK1H3H5,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1608186804795,0,True
1,5.0,Good. A bit slow,Nostalgic fun. A bit slow. I hope they don’t...,[],B00ZS80PC2,B07SRWRH5D,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1587051114941,1,False
2,5.0,... an order for my kids & they have really en...,This was an order for my kids & they have real...,[],B01FEHJYUU,B07MFMFW34,AGXVBIUFLFGMVLATYXHJYL4A5Q7Q,1490877431000,0,True
3,5.0,Great alt to pro controller,"These work great, They use batteries which is ...",[],B07GXJHRVK,B0BCHWZX95,AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q,1577637634017,0,True
4,5.0,solid product,I would recommend to anyone looking to add jus...,[],B00HUWA45W,B00HUWA45W,AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q,1427591932000,0,True


This dataset contains 4624615 records and 10 fields as listed below:

In [4]:
user_reviews.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 4624615 entries, 0 to 4624614
Data columns (total 10 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   rating             4624615 non-null  float64
 1   title              4624615 non-null  str    
 2   text               4624615 non-null  str    
 3   images             4624615 non-null  object 
 4   asin               4624615 non-null  str    
 5   parent_asin        4624615 non-null  str    
 6   user_id            4624615 non-null  str    
 7   timestamp          4624615 non-null  int64  
 8   helpful_vote       4624615 non-null  int64  
 9   verified_purchase  4624615 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1), str(5)
memory usage: 1.9+ GB


Each record contains a user review :

In [5]:
file_dict['user_reviews'][0]

{'rating': 4.0,
 'title': 'It’s pretty sexual. Not my fav',
 'text': 'I’m playing on ps5 and it’s interesting.  It’s unique, massive, and has a neat story.  People are freaking out angry about this game.  I don’t think it’s a top 10 game but it’s definitely a good game on ps5 (played at launch).',
 'images': [],
 'asin': 'B07DJWBYKP',
 'parent_asin': 'B07DK1H3H5',
 'user_id': 'AGCI7FAH4GL5FI65HYLKWTMFZ2CQ',
 'timestamp': 1608186804795,
 'helpful_vote': 0,
 'verified_purchase': True}

### Field selection

Field descriptions taken from :[https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-user-reviews](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-user-reviews)

|Field|	Type|Explanation|
|-----|----|------------|
|rating|float|	Rating of the product (from 1.0 to 5.0).|
|title|	str |	Title of the user review.|
|text	|str	|Text body of the user review.|
|images|	list|	Images that users post after they have received the product. Each image has different sizes (small, medium, large), represented by the small_image_url, medium_image_url, and large_image_url respectively.|
|asin	|str	|ID of the product.|
|parent_asin|	str	|Parent ID of the product. Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. Please use parent ID to find product meta.|
|user_id	|str|	ID of the reviewer|
|timestamp	|int|	Time of the review (unix time)|
|verified_purchase	|bool	|User purchase verification|
|helpful_vote|	int	|Helpful votes of the review|


If we only keep `verified_purchase=True`, we still have enough data points to make our analysis. This would ensure we only use true reviews which can help make the system more accurate.

In [6]:
user_reviews['verified_purchase'].value_counts()

verified_purchase
True     3982807
False     641808
Name: count, dtype: int64

Only 4% of records have images, we can remove the column.

In [7]:
round((user_reviews['images'].str.len() > 0).sum()/len(user_reviews['images'])*100,2)

np.float64(4.43)

We can see that 75% of records have a less than 1 helpful vote. This means that if we were to only consider the records that have a high number helpful votes, we would not have enough records for our analysis. 

In [8]:
user_reviews['helpful_vote'].describe()

count    4.624615e+06
mean     1.227983e+00
std      1.582127e+01
min     -1.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      1.036900e+04
Name: helpful_vote, dtype: float64

Useful fields:  
- `title` : review title can be used to get more information about the product for query search, 
- `text` : review can be used to get more information about the product for query search,  
- `parent_asin` : needed for tracking product across files , 
- `verified_purchas`e : filter data frame to only keep `verified_purchase=True` records, then remove field  

Less useful fields: 
- `rating` : can use average_rating from `meta_data`,
- `images` : many records don't contain images, 
- `asin` : information already given in `parent_asin` field,
- `timestamp` : the time of review doesn't add more information about products,
- `user_id` : the user id of review doesn't add more information about products,
- `helpful_vote` : not enough meaningful values for analysis.

#### Summary :
|Field| Keep |
|-----|-------|
|rating|No|
|title|	Yes|
|text	|Yes|
|images|No|
|asin	|No|
|parent_asin|Yes|
|user_id	|No|
|timestamp	|No|
|verified_purchase|Yes|
|helpful_vote|No|

## Item meta data 

### Description

In [9]:
meta_data = pd.DataFrame(file_dict['meta_data'])
meta_data.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Video Games,Dash 8-300 Professional Add-On,5.0,1,[Features Dash 8-300 and 8-Q300 ('Q' rollout l...,[The Dash 8-300 Professional Add-On lets you p...,None,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aerosoft,"[Video Games, PC, Games]",{'Pricing': 'The strikethrough price is the Li...,B000FH0MHO,None,NaN,NaN
1,Video Games,Phantasmagoria: A Puzzle of Flesh,4.1,18,[Windows 95],[],None,[{'thumb': 'https://m.media-amazon.com/images/...,[],Sierra,"[Video Games, PC, Games]","{'Best Sellers Rank': {'Video Games': 137612, ...",B00069EVOG,None,NaN,NaN
2,Video Games,NBA 2K17 - Early Tip Off Edition - PlayStation 4,4.3,223,[The #1 rated NBA video game simulation series...,[Following the record-breaking launch of NBA 2...,58.0,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'NBA 2K17 - Kobe: Haters vs Players...,2K,"[Video Games, PlayStation 4, Games]","{'Release date': 'September 16, 2016', 'Best S...",B00Z9TLVK0,None,NaN,NaN
3,Video Games,Nintendo Selects: The Legend of Zelda Ocarina ...,4.9,22,[Authentic Nintendo Selects: The Legend of Zel...,[],37.42,[{'thumb': 'https://m.media-amazon.com/images/...,[],Amazon Renewed,"[Video Games, Legacy Systems, Nintendo Systems...","{'Best Sellers Rank': {'Video Games': 51019, '...",B07SZJZV88,None,NaN,NaN
4,Video Games,Thrustmaster Elite Fitness Pack for Nintendo Wii,3.0,3,"[Includes (9) Total Accessories, Pedometer, Wi...",[The Thrustmaster Motion Plus Elite Fitness Pa...,None,[{'thumb': 'https://m.media-amazon.com/images/...,[],THRUSTMASTER,"[Video Games, Legacy Systems, Nintendo Systems...","{'Release date': 'November 1, 2009', 'Pricing'...",B002WH4ZJG,None,NaN,NaN


This dataset contains 137269 records and 16 fields as listed below:

In [10]:
meta_data.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 137269 entries, 0 to 137268
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   main_category    126234 non-null  str    
 1   title            137269 non-null  str    
 2   average_rating   137269 non-null  float64
 3   rating_number    137269 non-null  int64  
 4   features         137269 non-null  object 
 5   description      137269 non-null  object 
 6   price            62008 non-null   object 
 7   images           137269 non-null  object 
 8   videos           137269 non-null  object 
 9   store            132908 non-null  str    
 10  categories       137269 non-null  object 
 11  details          137269 non-null  object 
 12  parent_asin      137269 non-null  str    
 13  bought_together  0 non-null       object 
 14  subtitle         350 non-null     str    
 15  author           262 non-null     object 
dtypes: float64(1), int64(1), object(9), str(5)
memory

Each record contains information about a product :

In [11]:
file_dict['meta_data'][0]

{'main_category': 'Video Games',
 'title': 'Dash 8-300 Professional Add-On',
 'average_rating': 5.0,
 'rating_number': 1,
 'features': ["Features Dash 8-300 and 8-Q300 ('Q' rollout livery)",
  'Airlines - US Airways, South African Express, Bahamasair, Augsburg Airways, Lufthansa Cityline, British Airways (Union Jack), British European, FlyBe, Intersky, Wideroe, Iberia, Tyrolean, QantasLink, BWIA',
  'Airports include - London City, Frankfurt, Milan and Amsterdam Schipol',
  'Includes PSS PanelConfig and LoadEdit tools'],
 'description': ['The Dash 8-300 Professional Add-On lets you pilot a real commuter special. Fly two versions of the popular Dash 8-300 in a total of 17 different liveries. The Dash 8-300 is one of the most popular short-haul aircraft available and this superbly modelled version from acclaimed aircraft developers PSS is modelled in two versions with a total of 17 different liveries. The package also includes scenery for three European airports, tutorials, tutorial flig

### Field selection

Field descriptions taken from :[https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-user-reviews](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-item-metadata)

|Field|	Type|Explanation| 
|-----|----|------------|
|main_category	|str|	Main category (i.e., domain) of the product.| 
|title	|str	|Name of the product.|  
|average_rating|	float|	Rating of the product shown on the product page.| 
|rating_number|	int	|Number of ratings in the product.| 
|features	|list	|Bullet-point format features of the product.| 
|description|	list	|Description of the product.| 
|price	|float|Price in US dollars (at time of crawling).| 
|images|	list|	Images of the product. Each image has different sizes (thumb, large, hi_res). The “variant” field shows the position of image.| 
|videos|	list|	Videos of the product including title and url.| 
|store|	str	|Store name of the product.| 
|categories	|list	|Hierarchical categories of the product.| 
|details|	dict	|Product details, including materials, brand, sizes, etc.| 
|parent_asin	|str	|Parent ID of the product.| 
|bought_together	|list	|Recommended bundles from the websites.| 

The `main_category` field gives a broad description of what type of product we are looking at:

In [12]:
meta_data['main_category'].value_counts()

main_category
Video Games                     81255
Computers                       17235
All Electronics                 14816
Cell Phones & Accessories        3884
Toys & Games                     2733
Software                         1511
Industrial & Scientific          1079
Amazon Home                       737
Home Audio & Theater              443
Tools & Home Improvement          369
Office Products                   295
Sports & Outdoors                 244
Buy a Kindle                      220
Movies & TV                       197
Books                             196
Musical Instruments               154
All Beauty                        126
Camera & Photo                    117
Portable Audio & Accessories      112
Digital Music                     104
Health & Personal Care             95
Automotive                         85
AMAZON FASHION                     54
Pet Supplies                       38
Grocery                            36
Baby                               2

The `categories` field gives more precise information about the products :

In [13]:
meta_data['categories'].value_counts()

categories
[Video Games, PC, Games]                                                                                             17758
[]                                                                                                                   12637
[Video Games, PC, Accessories, Gaming Keyboards]                                                                      4736
[Video Games, PC, Accessories, Headsets]                                                                              3945
[Video Games, PC, Accessories, Gaming Mice]                                                                           3798
                                                                                                                     ...  
[Video Games, FREE One-Day Shipping on Select Video Games with your Discover Card]                                       1
[Video Games, Legacy Systems, PlayStation Systems, PlayStation 3, Accessories, Controllers, Music Controllers]           1
[Vide

Since `categories` and `main_category` give similar information, keeping only 1 seems reasonable. 

The `image` field contains mostly URLs or information that can't be used in our query search implementation.

In [14]:
meta_data[['images']][:5]

,images
0,[{'thumb': 'https://m.media-amazon.com/images/...
1,[{'thumb': 'https://m.media-amazon.com/images/...
2,[{'thumb': 'https://m.media-amazon.com/images/...
3,[{'thumb': 'https://m.media-amazon.com/images/...
4,[{'thumb': 'https://m.media-amazon.com/images/...


Only 21% of records have videos, and the field doesn't add information that is useful for our query search implementation.

In [15]:
print(round((meta_data['videos'].str.len() > 0).sum()/len(meta_data['videos'])*100,2),'% of records')
meta_data[['videos']][:5]

21.06 % of records


,videos
0,[]
1,[]
2,[{'title': 'NBA 2K17 - Kobe: Haters vs Players...
3,[]
4,[]


Not enough records contain values for the author and subtitle fields.

In [16]:
print(round(meta_data['author'].notna().sum()/len(meta_data['author'])*100,2),'% of records contain values for author field')
print(round(meta_data['subtitle'].notna().sum()/len(meta_data['subtitle'])*100,2),'% of records contain values for subtitle field')

0.19 % of records contain values for author field
0.25 % of records contain values for subtitle field


Useful fields: 
- `title` : product title can be used to get more information about the product for query search,  
- `average_rating` : needed for web app, 
- `rating_number` : rating can be used to get more information about the product for query search,   
- `features` : features can be used to get more information about the product for query search,  
- `description` : descriptions can be used to get more information about the product for query search,  
- `price` : price can be used to get more information about the product for query search,  
- `store` : store can be used to get more information about the product for query search,
- `details` : use only ESRB rating, item type, and date first available, the rest can either be found in other fields or doesn't add more information about products,
- `parent_asin` : needed for tracking product across files ,


Less useful fields: 
- `main_category`: `categories` fields gives similar and more relevant information about the product,
- `images` : field contains information that can't be used in our query search implementation (only consideres textual information), 
- `videos ` : many records don't contain videos, 
- `bought_together`: none of the records contain values for this field, 
- `author `: not enough records have non-nan values,
- `subtitle `: not enough records have non-nan values,


#### Summary

|Field|	 Keep |
|-----|-------|
|main_category	| No |
|title	| Yes |
|average_rating| Yes |
|rating_number|	Yes |
|features	|Yes |
|description|	Yes |
|price	|Yes |
|images|No |
|videos|No |
|store|Yes |
|categories	|Yes |
|details|	Yes |
|parent_asin|Yes |
|bought_together|No |

## Preprocessing

The preprocessing implementation can be found in the [src/preprocess.py](../src/preprocess.py) module.

The preprocessing steps are as follows :
- process the item meta data :
    - extract item type, ESRB rating and date first available from `details` field from the item meta data 
        - creating separate columns with the information allows for better retrieval during the query search
    - change the `price` field type to string 
        - the field contained a lot of None values which were causing issues during preprocessing
    - drop unused columns 
        - justification defined above
    - rename title `column` to `item_title`
        - both the user reviews and item meta data contain a `title` column, renaming them makes it clearer which is which
- process the user reviews :
    - keep only verified purchases (`verified_purchase==True`)
        - keeping only verified purchases can help ensure the reviews are accurate
    - drop unused columns 
        - justification defined above
    - rename title `column` to `review_title`
        - both the user reviews and item meta data contain a `title` column, renaming them makes it clearer which is which
    - group reviews by product
        - grouping by product and merging all the reviews will help for RAG
- merge the two datasets (user reviews and item meta data) 
    - each row represents a product which contains all the reviews
- preprocess the data
    - transform text to lowercase
    - removes all elements that aren't lowercase letters, numbers or whitespace
    - if the given text is a list, transforms it into a string